# 🫁 Chest X-Ray Pneumonia Detection v2
## EfficientNet-B0 (Classification) + YOLOv8-seg (Segmentation)

### Key design decisions
| Decision | Choice | Reason |
|---|---|---|
| Imbalance handling | **Undersample majority (PNEUMONIA)** to match NORMAL count | Avoids model bias; safer than oversampling artifacts |
| YOLO task | **YOLOv8-seg** (segmentation, not detection) | Pixel-level masks beat bounding boxes for diffuse infiltrates/consolidation |
| Pseudo-labels | **GradCAM polygon masks** from EfficientNet | No ground-truth segmentation labels exist in dataset |
| EfficientNet optimizer | **AdamW** (both phases) | Better L2 regularisation than plain Adam |
| YOLO Ph1 optimizer | **SGD + momentum + cosine LR** | Standard in YOLO literature for frozen-backbone phase |
| YOLO Ph2 optimizer | **AdamW + cosine LR** | Smooth fine-tune of all layers |

---
## Cell 1 — Install Dependencies

In [ ]:
!pip install -q kaggle ultralytics timm grad-cam \
    scikit-learn matplotlib seaborn opencv-python-headless \
    albumentations torchvision torch tqdm Pillow
print('✅ Dependencies installed')

---
## Cell 2 — Kaggle Auth & Dataset Download

In [ ]:
from google.colab import files
import os, shutil

print('Upload your kaggle.json (kaggle.com → Account → API → Create New Token)')
uploaded = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('✅ Kaggle credentials set')

In [ ]:
DATASET_SLUG = 'paultimothymooney/chest-xray-pneumonia'
DOWNLOAD_DIR = '/content/chest_xray_raw'
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

print(f'Downloading {DATASET_SLUG} ...')
!kaggle datasets download -d {DATASET_SLUG} -p {DOWNLOAD_DIR} --unzip

# Locate dataset root (contains train/ test/ val/)
DATASET_ROOT = None
for root, dirs, _ in os.walk(DOWNLOAD_DIR):
    if 'train' in dirs and 'test' in dirs:
        DATASET_ROOT = root
        break

assert DATASET_ROOT, 'Could not find dataset root — check the download'
print(f'\n✅ Dataset root: {DATASET_ROOT}')
for split in ['train', 'test', 'val']:
    for cls in ['NORMAL', 'PNEUMONIA']:
        p = os.path.join(DATASET_ROOT, split, cls)
        if os.path.exists(p):
            print(f'  {split}/{cls}: {len(os.listdir(p))} images')

---
## Cell 3 — Imports & Global Setup

In [ ]:
import os, random, shutil, warnings, time, json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve
)
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

---
## Cell 4 — EDA

In [ ]:
# ── Build master dataframe (pool ALL images — we re-split ourselves) ──────────
records = []
for split in ['train', 'test', 'val']:
    for cls in ['NORMAL', 'PNEUMONIA']:
        folder = Path(DATASET_ROOT) / split / cls
        if folder.exists():
            for p in folder.iterdir():
                if p.suffix.lower() in ['.jpeg', '.jpg', '.png']:
                    records.append({
                        'path': str(p),
                        'orig_split': split,
                        'label': cls,
                        'label_int': 0 if cls == 'NORMAL' else 1
                    })

df_all = pd.DataFrame(records)
print('=== Raw dataset (before balancing) ===')
print(df_all['label'].value_counts())
print(f'Total: {len(df_all)} images\n')
print('Original split breakdown:')
print(df_all.groupby(['orig_split','label']).size().unstack(fill_value=0))

In [ ]:
# ── EDA Plot 1: Raw class imbalance ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Class Imbalance — Before & After Undersampling', fontsize=14, fontweight='bold')

palette = {'NORMAL': '#2196F3', 'PNEUMONIA': '#F44336'}
counts_raw = df_all['label'].value_counts()

bars = axes[0].bar(counts_raw.index, counts_raw.values,
                   color=[palette[l] for l in counts_raw.index],
                   edgecolor='black', linewidth=0.8)
for bar, val in zip(bars, counts_raw.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
                 str(val), ha='center', fontweight='bold', fontsize=13)
axes[0].set_title('Before Undersampling', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)
n_normal   = counts_raw['NORMAL']
n_pneumo   = counts_raw['PNEUMONIA']
axes[0].annotate(
    f'Imbalance ratio\n{n_pneumo/n_normal:.2f}:1 (Pneumonia:Normal)',
    xy=(0.5, 0.85), xycoords='axes fraction', ha='center',
    fontsize=10, color='darkred',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFE0E0', edgecolor='red')
)

# After undersampling preview
balanced_counts = {'NORMAL': n_normal, 'PNEUMONIA': n_normal}
bars2 = axes[1].bar(balanced_counts.keys(), balanced_counts.values(),
                    color=[palette[l] for l in balanced_counts.keys()],
                    edgecolor='black', linewidth=0.8)
for bar, val in zip(bars2, balanced_counts.values()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 str(val), ha='center', fontweight='bold', fontsize=13)
axes[1].set_title('After Undersampling (balanced)', fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)
axes[1].annotate(
    f'Imbalance ratio\n1.00:1 (balanced)',
    xy=(0.5, 0.85), xycoords='axes fraction', ha='center',
    fontsize=10, color='darkgreen',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='#E0FFE0', edgecolor='green')
)

plt.tight_layout()
plt.savefig('/content/eda_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── EDA Plot 2: Sample images ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 6, figsize=(22, 8))
fig.suptitle('Sample Chest X-Ray Images', fontsize=15, fontweight='bold')

for row_idx, cls in enumerate(['NORMAL', 'PNEUMONIA']):
    subset = df_all[df_all['label'] == cls].sample(6, random_state=row_idx * 7)
    for col_idx, (_, row) in enumerate(subset.iterrows()):
        ax = axes[row_idx][col_idx]
        img = Image.open(row['path']).convert('L')  # grayscale for X-rays
        ax.imshow(img, cmap='gray')
        ax.set_title(cls, fontsize=9,
                     color='#F44336' if cls == 'PNEUMONIA' else '#2196F3',
                     fontweight='bold')
        ax.axis('off')

# Add row labels
axes[0][0].set_ylabel('NORMAL', fontsize=11, fontweight='bold', color='#2196F3')
axes[1][0].set_ylabel('PNEUMONIA', fontsize=11, fontweight='bold', color='#F44336')
plt.tight_layout()
plt.savefig('/content/eda_samples.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── EDA Plot 3: Image dimension & pixel intensity distributions ───────────────
print('Sampling 300 images for dimension/intensity analysis...')
sample_df = df_all.groupby('label').apply(
    lambda g: g.sample(min(150, len(g)), random_state=42)
).reset_index(drop=True)

widths, heights, means_n, means_p = [], [], [], []
for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc='Reading images'):
    try:
        with Image.open(row['path']) as img:
            w, h = img.size
            widths.append(w)
            heights.append(h)
            arr = np.array(img.convert('L').resize((224, 224)))
            if row['label'] == 'NORMAL':
                means_n.append(arr.mean())
            else:
                means_p.append(arr.mean())
    except Exception:
        pass

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle('Image Properties', fontsize=13, fontweight='bold')

axes[0].hist(widths,  bins=30, color='#2196F3', edgecolor='black', alpha=0.85)
axes[0].set_title('Width Distribution');  axes[0].set_xlabel('Pixels')

axes[1].hist(heights, bins=30, color='#4CAF50', edgecolor='black', alpha=0.85)
axes[1].set_title('Height Distribution'); axes[1].set_xlabel('Pixels')

axes[2].hist(means_n, bins=25, color='#2196F3', alpha=0.7, label='NORMAL',    edgecolor='black')
axes[2].hist(means_p, bins=25, color='#F44336', alpha=0.7, label='PNEUMONIA', edgecolor='black')
axes[2].axvline(np.mean(means_n), color='blue',  linestyle='--', lw=1.5, label=f'NORMAL mean={np.mean(means_n):.1f}')
axes[2].axvline(np.mean(means_p), color='red',   linestyle='--', lw=1.5, label=f'PNEUMONIA mean={np.mean(means_p):.1f}')
axes[2].set_title('Pixel Intensity (by Class)'); axes[2].set_xlabel('Mean Pixel Value')
axes[2].legend(fontsize=8)

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

print(f'Width  median: {np.median(widths):.0f}px | Height median: {np.median(heights):.0f}px')
print(f'NORMAL mean intensity:    {np.mean(means_n):.2f}')
print(f'PNEUMONIA mean intensity: {np.mean(means_p):.2f}')
plt.tight_layout()
plt.savefig('/content/eda_props.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Cell 5 — ⚙️ CONFIGURABLE HYPERPARAMETERS
> **Edit only this cell to change any experiment setting.**

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║            HYPERPARAMETERS  —  EDIT THIS CELL AS NEEDED                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── Reproducibility ───────────────────────────────────────────────────────────
RANDOM_SEED             = 42

# ── Class balancing ───────────────────────────────────────────────────────────
UNDERSAMPLE_MAJORITY    = True     # Shrink PNEUMONIA to match NORMAL count

# ── Dataset split ─────────────────────────────────────────────────────────────
TRAIN_RATIO             = 0.80
VAL_RATIO               = 0.10
TEST_RATIO              = 0.10

# ── Image sizes ───────────────────────────────────────────────────────────────
EFFNET_IMG_SIZE         = 224      # EfficientNet-B0 canonical
YOLO_IMG_SIZE           = 640      # YOLOv8 canonical

# ── ImageNet normalisation stats ──────────────────────────────────────────────
IMAGENET_MEAN           = [0.485, 0.456, 0.406]
IMAGENET_STD            = [0.229, 0.224, 0.225]

# ── DataLoader ────────────────────────────────────────────────────────────────
BATCH_SIZE              = 32
NUM_WORKERS             = 2
PIN_MEMORY              = True

# ── EfficientNet-B0 — Phase 1 (frozen backbone) ───────────────────────────────
EFFNET_FROZEN_EPOCHS    = 10
EFFNET_FROZEN_LR        = 1e-3
EFFNET_FROZEN_WD        = 1e-4     # AdamW weight decay

# ── EfficientNet-B0 — Phase 2 (full fine-tune) ────────────────────────────────
EFFNET_FT_EPOCHS        = 6
EFFNET_FT_LR            = 5e-5     # Lower LR for fine-tune
EFFNET_FT_WD            = 1e-4

# ── EfficientNet-B0 — Architecture ───────────────────────────────────────────
EFFNET_DROPOUT          = 0.35
EFFNET_LABEL_SMOOTHING  = 0.1      # CrossEntropyLoss label smoothing

# ── YOLOv8-seg config ─────────────────────────────────────────────────────────
YOLO_MODEL_VARIANT      = 'yolov8n-seg'   # n=nano (fast); swap to s/m for accuracy
YOLO_PRETRAINED_WEIGHTS = 'yolov8n-seg.pt'  # COCO pretrained segmentation weights

# ── YOLO Phase 1 (frozen backbone, SGD) ───────────────────────────────────────
YOLO_FROZEN_EPOCHS      = 10
YOLO_FROZEN_FREEZE_N    = 10       # Number of backbone layers to freeze
YOLO_FROZEN_LR0         = 1e-2    # SGD initial LR
YOLO_FROZEN_LRF         = 0.01    # Final LR as fraction of LR0 (cosine decay)
YOLO_FROZEN_MOMENTUM    = 0.937   # SGD momentum (standard YOLO default)
YOLO_FROZEN_WD          = 5e-4
YOLO_FROZEN_OPTIMIZER   = 'SGD'   # 'SGD' | 'Adam' | 'AdamW'

# ── YOLO Phase 2 (full fine-tune, AdamW) ──────────────────────────────────────
YOLO_FT_EPOCHS          = 5
YOLO_FT_LR0             = 1e-4    # AdamW initial LR (lower — fine-tune)
YOLO_FT_LRF             = 0.1
YOLO_FT_WD              = 1e-4
YOLO_FT_OPTIMIZER       = 'AdamW' # 'SGD' | 'Adam' | 'AdamW'

# ── YOLO shared ───────────────────────────────────────────────────────────────
YOLO_PATIENCE           = 5        # Early stopping patience
YOLO_CONF_THRESH        = 0.25
YOLO_IOU_THRESH         = 0.45
GRADCAM_PSEUDO_THRESH   = 0.40    # Heatmap threshold for mask polygon extraction

# ── Augmentation flags ────────────────────────────────────────────────────────
USE_AUGMENTATION        = True
AUG_FLIP_PROB           = 0.5
AUG_ROTATION_DEG        = 15
AUG_BRIGHTNESS          = 0.2
AUG_CONTRAST            = 0.2

# ── Class info ────────────────────────────────────────────────────────────────
NUM_CLASSES             = 2
CLASS_NAMES             = ['NORMAL', 'PNEUMONIA']

# ── Output paths ──────────────────────────────────────────────────────────────
OUTPUT_DIR              = Path('/content/outputs')
EFFNET_CKPT             = OUTPUT_DIR / 'efficientnet_b0_best.pth'
YOLO_DATA_YAML          = OUTPUT_DIR / 'yolo_seg_data.yaml'
YOLO_PROJECT_DIR        = OUTPUT_DIR / 'yolo_runs'

# ── Apply seeds ───────────────────────────────────────────────────────────────
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

print('✅ Hyperparameters configured')
print(f'   Undersample majority: {UNDERSAMPLE_MAJORITY}')
print(f'   EfficientNet: {EFFNET_FROZEN_EPOCHS} frozen (AdamW lr={EFFNET_FROZEN_LR}) '
      f'+ {EFFNET_FT_EPOCHS} finetune (AdamW lr={EFFNET_FT_LR})')
print(f'   YOLOv8-seg:   {YOLO_FROZEN_EPOCHS} frozen ({YOLO_FROZEN_OPTIMIZER} lr={YOLO_FROZEN_LR0}) '
      f'+ {YOLO_FT_EPOCHS} finetune ({YOLO_FT_OPTIMIZER} lr={YOLO_FT_LR0})')

---
## Cell 6 — Balance Dataset & 80/10/10 Split

In [ ]:
# ── Step 1: Undersample PNEUMONIA to match NORMAL count ──────────────────────
df_normal   = df_all[df_all['label'] == 'NORMAL']
df_pneumo   = df_all[df_all['label'] == 'PNEUMONIA']

if UNDERSAMPLE_MAJORITY:
    n_keep = len(df_normal)   # target count = size of minority class
    df_pneumo = df_pneumo.sample(n=n_keep, random_state=RANDOM_SEED)
    print(f'Undersampled PNEUMONIA: {len(df_all[df_all["label"]=="PNEUMONIA"])} → {n_keep}')

df_balanced = pd.concat([df_normal, df_pneumo]).sample(
    frac=1, random_state=RANDOM_SEED
).reset_index(drop=True)

print(f'\nBalanced dataset: {len(df_balanced)} images')
print(df_balanced['label'].value_counts())

# ── Step 2: Stratified 80/10/10 split ────────────────────────────────────────
all_paths  = df_balanced['path'].tolist()
all_labels = df_balanced['label_int'].tolist()

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_paths, all_labels,
    test_size=VAL_RATIO + TEST_RATIO,
    stratify=all_labels, random_state=RANDOM_SEED
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels,
    test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
    stratify=temp_labels, random_state=RANDOM_SEED
)

print(f'\n80/10/10 split (balanced):')
print(f'  Train : {len(train_paths)} | NORMAL={train_labels.count(0)}, PNEUMONIA={train_labels.count(1)}')
print(f'  Val   : {len(val_paths)}  | NORMAL={val_labels.count(0)},  PNEUMONIA={val_labels.count(1)}')
print(f'  Test  : {len(test_paths)}  | NORMAL={test_labels.count(0)},  PNEUMONIA={test_labels.count(1)}')

---
## Cell 7 — EfficientNet-B0: Dataset, Transforms, Model

In [ ]:
# ── Transforms ────────────────────────────────────────────────────────────────
train_tf = transforms.Compose([
    transforms.Resize((EFFNET_IMG_SIZE + 32, EFFNET_IMG_SIZE + 32)),
    transforms.RandomCrop(EFFNET_IMG_SIZE),
    transforms.RandomHorizontalFlip(p=AUG_FLIP_PROB),
    transforms.RandomRotation(AUG_ROTATION_DEG),
    transforms.ColorJitter(brightness=AUG_BRIGHTNESS, contrast=AUG_CONTRAST),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

eval_tf = transforms.Compose([
    transforms.Resize((EFFNET_IMG_SIZE, EFFNET_IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

# ── Dataset ───────────────────────────────────────────────────────────────────
class ChestXRayDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths; self.labels = labels; self.transform = transform

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

train_ds  = ChestXRayDataset(train_paths, train_labels, train_tf)
val_ds    = ChestXRayDataset(val_paths,   val_labels,   eval_tf)
test_ds   = ChestXRayDataset(test_paths,  test_labels,  eval_tf)

# No WeightedSampler needed — dataset is already balanced
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

print(f'✅ DataLoaders ready')
print(f'   Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

In [ ]:
# ── EfficientNet-B0 model ─────────────────────────────────────────────────────
class EfficientNetB0Classifier(nn.Module):
    def __init__(self, num_classes=2, dropout=0.35, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b0', pretrained=pretrained,
            num_classes=0, global_pool='avg'
        )
        feat_dim = self.backbone.num_features  # 1280
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(feat_dim),
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 512),
            nn.SiLU(),                         # SiLU (Swish) — matches EfficientNet activations
            nn.Dropout(dropout / 2),
            nn.Linear(512, num_classes)
        )

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False
        print('🔒 Backbone frozen')

    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = True
        print('🔓 Backbone unfrozen — full fine-tune')

    def forward(self, x):
        return self.classifier(self.backbone(x))

    @torch.no_grad()
    def predict_proba(self, x):
        return torch.softmax(self.forward(x), dim=1)


effnet = EfficientNetB0Classifier(
    num_classes=NUM_CLASSES, dropout=EFFNET_DROPOUT, pretrained=True
).to(DEVICE)

n_params = sum(p.numel() for p in effnet.parameters())
print(f'✅ EfficientNet-B0 loaded | Params: {n_params:,}')

---
## Cell 8 — EfficientNet Training (2-phase with AdamW)

In [ ]:
# ── Training loop utilities ───────────────────────────────────────────────────
def run_epoch(model, loader, optimizer, criterion, device, training=True):
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_probs, all_labels = [], [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, labels in tqdm(loader, leave=False,
                                 desc='  Train' if training else '  Val/Test'):
            imgs, labels = imgs.to(device), labels.to(device)
            if training: optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, labels)
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            probs = torch.softmax(out, dim=1)[:, 1]
            preds = out.argmax(dim=1)
            total_loss += loss.item() * imgs.size(0)
            correct    += (preds == labels).sum().item()
            total      += imgs.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / total, correct / total, all_preds, all_probs, all_labels


def train_phase(model, tr_loader, val_loader, optimizer, scheduler,
                criterion, epochs, device, history, phase_name):
    best_val_loss = float('inf')
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc, _, _, _       = run_epoch(model, tr_loader,  optimizer, criterion, device, True)
        vl_loss, vl_acc, _, _, _       = run_epoch(model, val_loader, optimizer, criterion, device, False)
        scheduler.step()
        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(vl_loss)
        history['val_acc'].append(vl_acc)
        history['phase'].append(phase_name)
        print(f'  [{phase_name}] Ep {ep:02d}/{epochs} | '
              f'loss {tr_loss:.4f}/{vl_loss:.4f} | '
              f'acc  {tr_acc:.3f}/{vl_acc:.3f} | '
              f'lr {scheduler.get_last_lr()[0]:.2e} | '
              f'{time.time()-t0:.1f}s')
        if vl_loss < best_val_loss:
            best_val_loss = vl_loss
            torch.save(model.state_dict(), EFFNET_CKPT)
            print(f'    💾 Saved (val_loss={vl_loss:.4f})')
    return history


criterion = nn.CrossEntropyLoss(label_smoothing=EFFNET_LABEL_SMOOTHING)
effnet_history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[], 'phase':[]}
print('✅ Training utilities ready')

In [ ]:
# ── Phase 1: Frozen backbone + AdamW ─────────────────────────────────────────
effnet.freeze_backbone()

opt_frozen = optim.AdamW(
    filter(lambda p: p.requires_grad, effnet.parameters()),
    lr=EFFNET_FROZEN_LR, weight_decay=EFFNET_FROZEN_WD
)
sch_frozen = optim.lr_scheduler.CosineAnnealingLR(opt_frozen, T_max=EFFNET_FROZEN_EPOCHS, eta_min=1e-6)

print(f'\n📌 EfficientNet Phase 1 — Frozen backbone ({EFFNET_FROZEN_EPOCHS} epochs, AdamW)')
effnet_history = train_phase(
    effnet, train_loader, val_loader, opt_frozen, sch_frozen,
    criterion, EFFNET_FROZEN_EPOCHS, DEVICE, effnet_history, 'Frozen'
)

In [ ]:
# ── Phase 2: Full fine-tune + AdamW (lower LR) ───────────────────────────────
effnet.unfreeze_backbone()

# Differential learning rates: backbone gets 10× lower LR than head
param_groups = [
    {'params': effnet.backbone.parameters(),    'lr': EFFNET_FT_LR / 10},
    {'params': effnet.classifier.parameters(),  'lr': EFFNET_FT_LR}
]
opt_ft = optim.AdamW(param_groups, weight_decay=EFFNET_FT_WD)
sch_ft = optim.lr_scheduler.CosineAnnealingLR(opt_ft, T_max=EFFNET_FT_EPOCHS, eta_min=1e-7)

print(f'\n🔓 EfficientNet Phase 2 — Full fine-tune ({EFFNET_FT_EPOCHS} epochs, AdamW, differential LR)')
print(f'   Backbone LR: {EFFNET_FT_LR/10:.2e} | Head LR: {EFFNET_FT_LR:.2e}')
effnet_history = train_phase(
    effnet, train_loader, val_loader, opt_ft, sch_ft,
    criterion, EFFNET_FT_EPOCHS, DEVICE, effnet_history, 'FineTune'
)

print('\n✅ EfficientNet training complete')

---
## Cell 9 — EfficientNet Evaluation Plots

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
h = effnet_history
n_ep = len(h['train_loss'])
xs   = range(1, n_ep + 1)
boundary = h['phase'].index('FineTune') if 'FineTune' in h['phase'] else n_ep

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('EfficientNet-B0 — Training Curves', fontsize=15, fontweight='bold')

for ax, key, title in [(ax1, 'loss', 'Loss'), (ax2, 'acc', 'Accuracy')]:
    ax.plot(xs, h[f'train_{key}'], 'b-o', ms=4, label='Train')
    ax.plot(xs, h[f'val_{key}'],   'r-s', ms=4, label='Val')
    ax.axvline(boundary + 0.5, color='gray', ls='--', lw=1.5, label='Unfreeze')
    ax.fill_betweenx([0,2], 0.5, boundary+0.5, alpha=0.05, color='orange')
    ax.fill_betweenx([0,2], boundary+0.5, n_ep+0.5, alpha=0.05, color='green')
    ax.set_xlabel('Epoch'); ax.set_ylabel(title); ax.set_title(title)
    ax.legend(); ax.set_xlim(0.5, n_ep+0.5)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax2.set_ylim(0, 1.05)

# Annotate phases
ax2.text(boundary/2 + 0.5, 0.05, 'Frozen\nbackbone',
         ha='center', color='darkorange', fontsize=9, fontweight='bold')
ax2.text(boundary + (n_ep-boundary)/2 + 0.5, 0.05, 'Full\nfine-tune',
         ha='center', color='darkgreen', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('/content/effnet_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Load best checkpoint & test-set evaluation ────────────────────────────────
effnet.load_state_dict(torch.load(EFFNET_CKPT))
effnet.eval()

_, test_acc, test_preds, test_probs, test_true = run_epoch(
    effnet, test_loader, None, nn.CrossEntropyLoss(), DEVICE, training=False
)

test_f1   = f1_score(test_true,  test_preds, average='binary')
test_prec = precision_score(test_true, test_preds, zero_division=0)
test_rec  = recall_score(test_true,  test_preds, zero_division=0)
test_auc  = roc_auc_score(test_true,  test_probs)

print('\n📋 EfficientNet-B0 — Test Results')
print('=' * 50)
print(f'  Accuracy  : {test_acc:.4f}')
print(f'  Precision : {test_prec:.4f}')
print(f'  Recall    : {test_rec:.4f}')
print(f'  F1        : {test_f1:.4f}')
print(f'  ROC-AUC   : {test_auc:.4f}')
print()
print(classification_report(test_true, test_preds, target_names=CLASS_NAMES))

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
def plot_cm(y_true, y_pred, class_names, title, save_path=None):
    cm      = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=13, fontweight='bold')
    for ax, data, fmt, sub in [
        (axes[0], cm,      'd',    'Raw Counts'),
        (axes[1], cm_norm, '.2f',  'Normalized')
    ]:
        sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                    xticklabels=class_names, yticklabels=class_names,
                    ax=ax, linewidths=0.5, cbar_kws={'shrink': 0.8})
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
        ax.set_title(sub)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

plot_cm(test_true, test_preds, CLASS_NAMES,
        'EfficientNet-B0 — Confusion Matrix',
        '/content/effnet_cm.png')

In [ ]:
# ── F1/Prec/Rec bar + ROC curve ───────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('EfficientNet-B0 — Metrics', fontsize=14, fontweight='bold')

metrics_vals  = [test_acc, test_prec, test_rec, test_f1]
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1']
colors        = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
bars = ax1.bar(metrics_names, metrics_vals, color=colors, edgecolor='black')
for bar, val in zip(bars, metrics_vals):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
             f'{val:.3f}', ha='center', fontweight='bold')
ax1.set_ylim(0, 1.15); ax1.set_ylabel('Score')
ax1.set_title('Classification Metrics')
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

fpr, tpr, _ = roc_curve(test_true, test_probs)
ax2.plot(fpr, tpr, '#2196F3', lw=2, label=f'ROC AUC={test_auc:.3f}')
ax2.plot([0,1],[0,1],'k--', lw=1)
ax2.fill_between(fpr, tpr, alpha=0.1, color='#2196F3')
ax2.set_xlabel('False Positive Rate'); ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curve'); ax2.legend()
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('/content/effnet_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Cell 10 — GradCAM Visualisation (EfficientNet)

In [ ]:
try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    from pytorch_grad_cam.utils.image import show_cam_on_image
except ImportError:
    !pip install -q grad-cam
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    from pytorch_grad_cam.utils.image import show_cam_on_image

# Target last conv block of EfficientNet backbone
target_layers = [effnet.backbone.conv_head]
cam = GradCAM(model=effnet, target_layers=target_layers)

pneumo_idxs = [i for i, l in enumerate(test_labels) if l == 1][:4]

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
fig.suptitle(
    'GradCAM — Pneumonia Region Localisation\n'
    '(highlights bronchial infiltrates / consolidation patterns)',
    fontsize=13, fontweight='bold'
)

for col, idx in enumerate(pneumo_idxs):
    raw = Image.open(test_paths[idx]).convert('RGB').resize((EFFNET_IMG_SIZE, EFFNET_IMG_SIZE))
    raw_np = np.array(raw) / 255.0
    inp = eval_tf(Image.open(test_paths[idx]).convert('RGB')).unsqueeze(0).to(DEVICE)

    gc = cam(input_tensor=inp, targets=[ClassifierOutputTarget(1)])[0]
    cam_img = show_cam_on_image(raw_np.astype(np.float32), gc, use_rgb=True)

    # Threshold mask
    mask = (gc > GRADCAM_PSEUDO_THRESH).astype(np.uint8) * 255

    with torch.no_grad():
        prob = effnet.predict_proba(inp)[0, 1].item()

    axes[0][col].imshow(raw, cmap='gray')
    axes[0][col].set_title(f'Original #{idx}', fontsize=9)
    axes[0][col].axis('off')

    axes[1][col].imshow(cam_img)
    axes[1][col].set_title(f'GradCAM overlay\nP(pneumonia)={prob:.1%}', fontsize=9, color='red')
    axes[1][col].axis('off')

    axes[2][col].imshow(mask, cmap='hot')
    axes[2][col].set_title(f'Pseudo-mask (thresh={GRADCAM_PSEUDO_THRESH})', fontsize=9)
    axes[2][col].axis('off')

plt.tight_layout()
plt.savefig('/content/effnet_gradcam.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ GradCAM saved')

---
## Cell 11 — Build YOLOv8-seg Dataset (GradCAM Polygon Pseudo-Labels)

> **Why segmentation labels?** YOLO-seg requires polygon contours (normalised x,y pairs) instead of bounding boxes. We derive these by thresholding the GradCAM heatmap and extracting the largest contour.

In [ ]:
YOLO_DS = OUTPUT_DIR / 'yolo_seg_dataset'
for split in ['train', 'val', 'test']:
    (YOLO_DS / split / 'images').mkdir(parents=True, exist_ok=True)
    (YOLO_DS / split / 'labels').mkdir(parents=True, exist_ok=True)


def gradcam_to_seg_polygon(gc_heatmap, threshold=GRADCAM_PSEUDO_THRESH,
                            min_area_frac=0.005, n_points=20):
    """
    Convert a GradCAM heatmap to a YOLOv8-seg polygon label.

    Returns normalised flat list [x0,y0,x1,y1,...] or None if nothing found.
    """
    H, W = gc_heatmap.shape
    mask  = (gc_heatmap > threshold).astype(np.uint8)

    # Morphological cleanup
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None

    # Keep only the largest contour above minimum area fraction
    c     = max(contours, key=cv2.contourArea)
    area  = cv2.contourArea(c)
    if area < min_area_frac * H * W:
        return None

    # Approximate to n_points polygon
    epsilon = 0.02 * cv2.arcLength(c, True)
    approx  = cv2.approxPolyDP(c, epsilon, True)
    pts     = approx[:, 0, :]            # shape (K, 2)

    # Normalise to [0, 1]
    pts_norm = pts.astype(float)
    pts_norm[:, 0] /= W
    pts_norm[:, 1] /= H
    pts_norm = np.clip(pts_norm, 0.0, 1.0)

    return pts_norm.flatten().tolist()   # [x0,y0,x1,y1,...]


print('Generating YOLOv8-seg dataset with GradCAM polygon pseudo-labels...')

split_map = [
    ('train', train_paths, train_labels),
    ('val',   val_paths,   val_labels),
    ('test',  test_paths,  test_labels)
]

label_stats = {'with_mask': 0, 'empty': 0}

for split_name, paths, labels_list in split_map:
    print(f'\n  {split_name} ({len(paths)} images)')
    for img_path, label in tqdm(zip(paths, labels_list), total=len(paths),
                                desc=f'  {split_name}', leave=False):
        stem    = Path(img_path).stem
        dst_img = YOLO_DS / split_name / 'images' / f'{stem}.jpg'
        dst_lbl = YOLO_DS / split_name / 'labels' / f'{stem}.txt'

        # Resize & save image
        img_pil = Image.open(img_path).convert('RGB').resize((YOLO_IMG_SIZE, YOLO_IMG_SIZE))
        img_pil.save(dst_img, quality=92)

        if label == 1:   # PNEUMONIA
            inp = eval_tf(Image.open(img_path).convert('RGB')).unsqueeze(0).to(DEVICE)
            gc  = cam(input_tensor=inp, targets=[ClassifierOutputTarget(1)])[0]
            polygon = gradcam_to_seg_polygon(gc)

            if polygon and len(polygon) >= 6:  # YOLO needs at least 3 points
                coords_str = ' '.join(f'{v:.6f}' for v in polygon)
                with open(dst_lbl, 'w') as f:
                    f.write(f'0 {coords_str}\n')  # class 0 = pneumonia_region
                label_stats['with_mask'] += 1
            else:
                # Fallback: centre bounding box as 4-point polygon
                with open(dst_lbl, 'w') as f:
                    f.write('0 0.2 0.2 0.8 0.2 0.8 0.8 0.2 0.8\n')
                label_stats['with_mask'] += 1
        else:             # NORMAL — empty label (background)
            dst_lbl.touch()
            label_stats['empty'] += 1

print(f'\n✅ YOLO-seg dataset created')
print(f'   Pneumonia (with polygon mask): {label_stats["with_mask"]}')
print(f'   Normal (empty label files)  : {label_stats["empty"]}')

In [ ]:
# ── Write data.yaml ───────────────────────────────────────────────────────────
yaml_str = f"""path: {YOLO_DS}
train: train/images
val:   val/images
test:  test/images

nc: 1
names: ['pneumonia_region']
"""
with open(YOLO_DATA_YAML, 'w') as f:
    f.write(yaml_str)
print(f'✅ data.yaml written:\n{yaml_str}')

---
## Cell 12 — YOLOv8-seg Training
### Phase 1: SGD + momentum (frozen backbone) → Phase 2: AdamW (full fine-tune)

In [ ]:
from ultralytics import YOLO

# ── Phase 1: frozen backbone, SGD ────────────────────────────────────────────
yolo = YOLO(YOLO_PRETRAINED_WEIGHTS)   # COCO pretrained seg weights

print(f'\n📌 YOLOv8-seg Phase 1 — Frozen backbone ({YOLO_FROZEN_EPOCHS} epochs)')
print(f'   Optimizer : {YOLO_FROZEN_OPTIMIZER}  lr0={YOLO_FROZEN_LR0}  momentum={YOLO_FROZEN_MOMENTUM}')
print(f'   Freeze    : first {YOLO_FROZEN_FREEZE_N} backbone layers')

yolo.train(
    data      = str(YOLO_DATA_YAML),
    epochs    = YOLO_FROZEN_EPOCHS,
    imgsz     = YOLO_IMG_SIZE,
    batch     = BATCH_SIZE,
    optimizer = YOLO_FROZEN_OPTIMIZER,
    lr0       = YOLO_FROZEN_LR0,
    lrf       = YOLO_FROZEN_LRF,
    momentum  = YOLO_FROZEN_MOMENTUM,
    weight_decay = YOLO_FROZEN_WD,
    freeze    = YOLO_FROZEN_FREEZE_N,
    patience  = YOLO_PATIENCE,
    project   = str(YOLO_PROJECT_DIR),
    name      = 'phase1_sgd_frozen',
    exist_ok  = True,
    device    = 0 if torch.cuda.is_available() else 'cpu',
    workers   = NUM_WORKERS,
    conf      = YOLO_CONF_THRESH,
    iou       = YOLO_IOU_THRESH,
    augment   = USE_AUGMENTATION,
    fliplr    = AUG_FLIP_PROB,
    degrees   = AUG_ROTATION_DEG,
    hsv_v     = AUG_BRIGHTNESS,
    verbose   = True,
    task      = 'segment'
)

print('✅ YOLO Phase 1 complete')
YOLO_P1_BEST = str(YOLO_PROJECT_DIR / 'phase1_sgd_frozen' / 'weights' / 'best.pt')

In [ ]:
# ── Phase 2: full fine-tune, AdamW ───────────────────────────────────────────
yolo_ft = YOLO(YOLO_P1_BEST)

print(f'\n🔓 YOLOv8-seg Phase 2 — Full fine-tune ({YOLO_FT_EPOCHS} epochs)')
print(f'   Optimizer : {YOLO_FT_OPTIMIZER}  lr0={YOLO_FT_LR0}  (cosine decay → lrf×lr0)')

yolo_ft.train(
    data      = str(YOLO_DATA_YAML),
    epochs    = YOLO_FT_EPOCHS,
    imgsz     = YOLO_IMG_SIZE,
    batch     = BATCH_SIZE,
    optimizer = YOLO_FT_OPTIMIZER,
    lr0       = YOLO_FT_LR0,
    lrf       = YOLO_FT_LRF,
    weight_decay = YOLO_FT_WD,
    freeze    = 0,                    # unfreeze all
    patience  = YOLO_PATIENCE,
    project   = str(YOLO_PROJECT_DIR),
    name      = 'phase2_adamw_ft',
    exist_ok  = True,
    device    = 0 if torch.cuda.is_available() else 'cpu',
    workers   = NUM_WORKERS,
    conf      = YOLO_CONF_THRESH,
    iou       = YOLO_IOU_THRESH,
    augment   = USE_AUGMENTATION,
    verbose   = True,
    task      = 'segment'
)

YOLO_BEST = str(YOLO_PROJECT_DIR / 'phase2_adamw_ft' / 'weights' / 'best.pt')
print(f'\n✅ YOLO full training complete | Best weights: {YOLO_BEST}')

---
## Cell 13 — YOLOv8-seg Evaluation Plots

In [ ]:
# ── Load both phases and concat results.csv ───────────────────────────────────
def safe_load_results(path):
    df_r = pd.read_csv(path)
    df_r.columns = df_r.columns.str.strip()
    return df_r

df_p1 = safe_load_results(YOLO_PROJECT_DIR / 'phase1_sgd_frozen'  / 'results.csv')
df_p2 = safe_load_results(YOLO_PROJECT_DIR / 'phase2_adamw_ft'    / 'results.csv')
df_yolo = pd.concat([df_p1, df_p2], ignore_index=True)
df_yolo['epoch_global'] = range(1, len(df_yolo) + 1)
p1_len = len(df_p1)

print('YOLO result columns:', df_yolo.columns.tolist())

In [ ]:
# ── YOLO training curves ──────────────────────────────────────────────────────
metric_cols = [
    ('train/box_loss',         'Train Box Loss',        '#2196F3'),
    ('train/seg_loss',         'Train Seg Loss',        '#F44336'),
    ('train/cls_loss',         'Train Cls Loss',        '#9C27B0'),
    ('val/box_loss',           'Val Box Loss',          '#4CAF50'),
    ('metrics/precision(M)',   'Precision (mask)',      '#FF9800'),
    ('metrics/mAP50-95(M)',    'mAP50-95 (mask)',       '#795548'),
]

available = [(c, l, col) for c, l, col in metric_cols if c in df_yolo.columns]

n_plots = len(available)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f'YOLOv8-seg ({YOLO_MODEL_VARIANT}) — Training Curves\n'
             f'Phase 1: SGD (orange) → Phase 2: AdamW (green)',
             fontsize=13, fontweight='bold')

for ax, (col, lbl, color) in zip(axes.flat, available):
    ax.plot(df_yolo['epoch_global'], df_yolo[col], color=color, lw=2, marker='o', ms=3)
    ax.axvline(p1_len + 0.5, color='gray', ls='--', lw=1.5, label='Unfreeze (AdamW)')
    ax.fill_betweenx([df_yolo[col].min(), df_yolo[col].max()],
                     0.5, p1_len + 0.5, alpha=0.06, color='orange')
    ax.fill_betweenx([df_yolo[col].min(), df_yolo[col].max()],
                     p1_len + 0.5, len(df_yolo) + 0.5, alpha=0.06, color='green')
    ax.set_title(lbl, fontweight='bold'); ax.set_xlabel('Epoch')
    ax.legend(fontsize=8)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

for ax in axes.flat[len(available):]:
    ax.axis('off')

plt.tight_layout()
plt.savefig('/content/yolo_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── YOLO classification confusion matrix on test set ──────────────────────────
# (presence of any seg mask → PNEUMONIA, absence → NORMAL)
final_yolo = YOLO(YOLO_BEST)
yolo_preds, yolo_true = [], []

n_eval = min(300, len(test_paths))
print(f'Running YOLO inference on {n_eval} test images...')
for img_p, true_lbl in tqdm(zip(test_paths[:n_eval], test_labels[:n_eval]),
                            total=n_eval, desc='YOLO inference'):
    res   = final_yolo(img_p, verbose=False, conf=YOLO_CONF_THRESH)[0]
    pred  = 1 if (res.masks is not None and len(res.masks) > 0) else 0
    yolo_preds.append(pred)
    yolo_true.append(true_lbl)

print('\n📋 YOLOv8-seg — Classification Metrics')
print('=' * 50)
print(f'  Precision : {precision_score(yolo_true, yolo_preds, zero_division=0):.4f}')
print(f'  Recall    : {recall_score(yolo_true, yolo_preds, zero_division=0):.4f}')
print(f'  F1        : {f1_score(yolo_true, yolo_preds, average="binary", zero_division=0):.4f}')
print()
print(classification_report(yolo_true, yolo_preds, target_names=CLASS_NAMES, zero_division=0))

plot_cm(yolo_true, yolo_preds, CLASS_NAMES,
        'YOLOv8-seg — Confusion Matrix', '/content/yolo_cm.png')

---
## Cell 14 — YOLOv8-seg Mask Visualisation

In [ ]:
# ── Visualise segmentation masks on pneumonia test images ─────────────────────
pneumo_paths = [p for p, l in zip(test_paths, test_labels) if l == 1][:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle(
    'YOLOv8-seg — Pneumonia Region Segmentation Masks\n'
    '(pixel-level delineation of bronchial/alveolar infiltrates)',
    fontsize=13, fontweight='bold'
)

for ax, img_path in zip(axes.flat, pneumo_paths):
    res = final_yolo(img_path, verbose=False, conf=YOLO_CONF_THRESH)[0]

    img_bgr  = cv2.imread(img_path)
    img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    overlay  = img_rgb.copy()

    n_masks = 0
    if res.masks is not None:
        for mask_data in res.masks.xy:       # polygon contours
            pts = mask_data.astype(np.int32)
            # Scale to image size
            scale_x = img_bgr.shape[1] / YOLO_IMG_SIZE
            scale_y = img_bgr.shape[0] / YOLO_IMG_SIZE
            pts_scaled = pts.copy()
            pts_scaled[:, 0] = (pts[:, 0] * scale_x).astype(int)
            pts_scaled[:, 1] = (pts[:, 1] * scale_y).astype(int)
            cv2.fillPoly(overlay, [pts_scaled], (220, 50, 50))
            cv2.polylines(overlay, [pts_scaled], True, (255, 100, 100), 2)
            n_masks += 1

    # Alpha blend
    blended = cv2.addWeighted(img_rgb, 0.55, overlay, 0.45, 0)

    # EfficientNet confidence
    with torch.no_grad():
        t    = eval_tf(Image.open(img_path).convert('RGB')).unsqueeze(0).to(DEVICE)
        prob = effnet.predict_proba(t)[0, 1].item()

    ax.imshow(blended)
    ax.set_title(
        f'Seg masks: {n_masks} region(s)\n'
        f'EfficientNet P(pneumonia): {prob:.1%}',
        fontsize=9, color='darkred'
    )
    ax.axis('off')

plt.tight_layout()
plt.savefig('/content/yolo_seg_masks.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Segmentation masks visualisation saved')

---
## Cell 15 — Combined Inference Pipeline

In [ ]:
def predict(image_path: str, show: bool = True) -> dict:
    """
    Full combined inference: EfficientNet probability + YOLOv8-seg masks.

    Args:
        image_path : path to chest X-ray image
        show       : whether to render matplotlib figure

    Returns dict:
        pneumonia_probability  float
        diagnosis              'PNEUMONIA' or 'NORMAL'
        n_seg_masks            int
        seg_polygons           list of np.ndarray (pixel coords)
    """
    # ── EfficientNet ──────────────────────────────────────────────────────────
    img_pil = Image.open(image_path).convert('RGB')
    inp     = eval_tf(img_pil).unsqueeze(0).to(DEVICE)
    probs   = effnet.predict_proba(inp)[0].cpu().numpy()
    p_pneu  = float(probs[1])
    diag    = CLASS_NAMES[int(probs.argmax())]

    # ── YOLOv8-seg ────────────────────────────────────────────────────────────
    res      = final_yolo(image_path, verbose=False, conf=YOLO_CONF_THRESH)[0]
    polygons = []
    if res.masks is not None:
        polygons = [m.astype(np.int32) for m in res.masks.xy]

    # ── Visualisation ─────────────────────────────────────────────────────────
    if show:
        img_bgr  = cv2.imread(image_path)
        img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        overlay  = img_rgb.copy()
        for pts in polygons:
            sx = img_bgr.shape[1] / YOLO_IMG_SIZE
            sy = img_bgr.shape[0] / YOLO_IMG_SIZE
            sc = pts.copy(); sc[:,0] = (pts[:,0]*sx).astype(int); sc[:,1] = (pts[:,1]*sy).astype(int)
            cv2.fillPoly(overlay, [sc], (220, 50, 50))
            cv2.polylines(overlay, [sc], True, (255, 120, 120), 2)
        blended = cv2.addWeighted(img_rgb, 0.55, overlay, 0.45, 0)

        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        title_color = '#D32F2F' if diag == 'PNEUMONIA' else '#388E3C'
        fig.suptitle(f'Diagnosis: {diag}  |  P(pneumonia) = {p_pneu:.1%}',
                     fontsize=14, fontweight='bold', color=title_color)

        axes[0].imshow(img_rgb)
        axes[0].set_title('Original X-Ray', fontweight='bold'); axes[0].axis('off')

        axes[1].imshow(blended)
        axes[1].set_title(f'YOLOv8-seg Masks  ({len(polygons)} region(s))',
                          fontweight='bold'); axes[1].axis('off')

        # Probability gauge
        gauge_ax = axes[2]
        gauge_ax.barh(['Normal', 'Pneumonia'], [1 - p_pneu, p_pneu],
                      color=['#2196F3', '#F44336'], edgecolor='black')
        gauge_ax.set_xlim(0, 1)
        gauge_ax.axvline(0.5, color='black', ls='--', lw=1)
        for i, v in enumerate([1 - p_pneu, p_pneu]):
            gauge_ax.text(min(v + 0.02, 0.97), i, f'{v:.1%}',
                          va='center', fontweight='bold')
        gauge_ax.set_xlabel('Probability')
        gauge_ax.set_title('EfficientNet Confidence', fontweight='bold')
        gauge_ax.spines['top'].set_visible(False)
        gauge_ax.spines['right'].set_visible(False)

        plt.tight_layout()
        plt.show()

    return {
        'pneumonia_probability': p_pneu,
        'normal_probability':    float(probs[0]),
        'diagnosis':             diag,
        'n_seg_masks':           len(polygons),
        'seg_polygons':          polygons
    }


print('✅ predict() ready')
print('Usage:  result = predict("/path/to/xray.jpg")')

In [ ]:
# ── Demo ─────────────────────────────────────────────────────────────────────
demo_pneu   = [p for p, l in zip(test_paths, test_labels) if l == 1][0]
demo_normal = [p for p, l in zip(test_paths, test_labels) if l == 0][0]

print('--- PNEUMONIA sample ---')
r1 = predict(demo_pneu)

print('\n--- NORMAL sample ---')
r2 = predict(demo_normal)

---
## Cell 16 — Save Models & Print Summary

In [ ]:
# ── Save models ───────────────────────────────────────────────────────────────
final_effnet_path = OUTPUT_DIR / 'efficientnet_b0_pneumonia.pth'
final_yolo_path   = OUTPUT_DIR / 'yolov8seg_pneumonia.pt'

torch.save({
    'model_state_dict': effnet.state_dict(),
    'class_names':  CLASS_NAMES,
    'img_size':     EFFNET_IMG_SIZE,
    'mean':         IMAGENET_MEAN,
    'std':          IMAGENET_STD,
    'best_val_acc': max(effnet_history['val_acc'])
}, final_effnet_path)

shutil.copy(YOLO_BEST, final_yolo_path)

print('✅ Models saved')
print(f'   EfficientNet-B0 : {final_effnet_path}')
print(f'   YOLOv8-seg      : {final_yolo_path}')

print('\n' + '='*62)
print('                   EXPERIMENT SUMMARY')
print('='*62)
print(f'Dataset        : Chest X-Ray Pneumonia (Kaggle, pooled+replit)')
print(f'Imbalance fix  : Random undersample PNEUMONIA → {len(df_normal)} each class')
print(f'Balanced total : {len(df_balanced)} | Split {int(TRAIN_RATIO*100)}/{int(VAL_RATIO*100)}/{int(TEST_RATIO*100)}')
print()
print(f'[EfficientNet-B0]  Pretrained: ImageNet (timm)')
print(f'  Phase 1: {EFFNET_FROZEN_EPOCHS} ep | AdamW lr={EFFNET_FROZEN_LR} | Backbone frozen')
print(f'  Phase 2: {EFFNET_FT_EPOCHS} ep | AdamW lr={EFFNET_FT_LR} (diff. lr) | Full fine-tune')
print(f'  Test Acc={test_acc:.4f} | F1={test_f1:.4f} | AUC={test_auc:.4f}')
print()
print(f'[YOLOv8-seg {YOLO_MODEL_VARIANT}]  Pretrained: COCO seg')
print(f'  Phase 1: {YOLO_FROZEN_EPOCHS} ep | {YOLO_FROZEN_OPTIMIZER} lr={YOLO_FROZEN_LR0} mom={YOLO_FROZEN_MOMENTUM} | freeze={YOLO_FROZEN_FREEZE_N}')
print(f'  Phase 2: {YOLO_FT_EPOCHS} ep | {YOLO_FT_OPTIMIZER} lr={YOLO_FT_LR0} | Fully unfrozen')
print(f'  Task: pixel-level pneumonia region segmentation (GradCAM pseudo-labels)')
print('='*62)